In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import copy
import json

import numpy as np
import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AlbertForSequenceClassification,
    AlbertTokenizer,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import f1_score

from tqdm.auto import tqdm

device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")
model     = AlbertForSequenceClassification.from_pretrained("albert-base-v2", num_labels=3)
model     = model.to(device)

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
LABEL2ID = {"e": 0, "n": 1, "c": 2}
BASE     = "drive/MyDrive/anli/"

def load_jsonl(filepath):
    with open(filepath) as f:
        return [json.loads(l) for l in open(filepath) if l.strip()]

class ANLIDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        row = self.data[idx]
        enc = tokenizer(row["context"], row["hypothesis"],
                        max_length=128, truncation=True,
                        padding="max_length", return_tensors="pt")
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(LABEL2ID[row["label"]], dtype=torch.long)
        }

r1_data = load_jsonl(BASE + "R1/" + "train.jsonl")
r2_data = load_jsonl(BASE + "R2/" + "train.jsonl")
r3_data = load_jsonl(BASE + "R3/" + "train.jsonl")
r1_dev  = load_jsonl(BASE + "R1/" + "dev.jsonl")
r2_dev  = load_jsonl(BASE + "R2/" + "dev.jsonl")
r3_dev  = load_jsonl(BASE + "R3/" + "dev.jsonl")

print("R1:", len(r1_data), "| R2:", len(r2_data), "| R3:", len(r3_data))

R1: 16946 | R2: 45460 | R3: 100459


In [4]:
def evaluate(model, data, device):
    model.eval()

    loader = DataLoader(
        ANLIDataset(data),
        batch_size=32,
        shuffle=False
    )

    all_preds = []
    all_labels = []
    total_loss = 0

    with torch.no_grad():
        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            preds = outputs.logits.argmax(dim=-1)

            total_loss += outputs.loss.item()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_labels))

    f1 = f1_score(
        all_labels,
        all_preds,
        average=None,
        labels=[0, 1, 2]
    )

    avg_loss = total_loss / len(loader)

    return avg_loss, acc, f1

In [ ]:
# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Store results for graphs
# -------------------------
history = {
    "R1": [],
    "R2": [],
    "R3": []
}

# -------------------------
# Curriculum Learning
# -------------------------
for round_name, train_data, val_data in [

    ("R1", r1_data, r1_dev),
    ("R2", r2_data, r2_dev),
    ("R3", r3_data, r3_dev),

]:

    print(f"\n{'='*12} {round_name} {'='*12}")

    train_loader = DataLoader(
        ANLIDataset(train_data),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    total_steps = len(train_loader) * MAX_EPOCHS
    warmup_steps = int(0.1 * total_steps)

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_val_loss = float("inf")
    best_model_state = None

    checkpoint_path = f"/content/best_{round_name}"

    print(
        f"{'Epoch':<8}"
        f"{'Train Loss':<15}"
        f"{'Val Loss':<15}"
        f"{'Accuracy':<12}"
        f"{'F1 Entail':<15}"
        f"{'F1 Neutral':<15}"
        f"{'F1 Contra'}"
    )

    # ==========================
    # Epoch Loop
    # ==========================
    for epoch in range(MAX_EPOCHS):

        model.train()

        running_loss = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"{round_name} Epoch {epoch+1}/{MAX_EPOCHS}",
            dynamic_ncols=True
        )

        for batch in progress_bar:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            running_loss += loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}"
            )

        # ==========================
        # End of Epoch Evaluation
        # ==========================

        train_loss = running_loss / len(train_loader)

        val_loss, accuracy, f1 = evaluate(
            model,
            val_data,
            device
        )

        # Store metrics for graphs
        history[round_name].append({

            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "accuracy": accuracy,
            "f1_entailment": f1[0],
            "f1_neutral": f1[1],
            "f1_contradiction": f1[2]

        })

        print(
            f"{epoch+1:<8}"
            f"{train_loss:<15.6f}"
            f"{val_loss:<15.6f}"
            f"{accuracy:<12.6f}"
            f"{f1[0]:<15.6f}"
            f"{f1[1]:<15.6f}"
            f"{f1[2]:.6f}"
        )

        # ==========================
        # Save Best Model
        # ==========================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_state = copy.deepcopy(model.state_dict())

            model.save_pretrained(checkpoint_path)
            tokenizer.save_pretrained(checkpoint_path)

            print("✓ Best model updated.")

    # ==========================
    # Restore Best Model
    # ==========================

    print(f"\nLoading best checkpoint from {round_name}")

    model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

model.save_pretrained("/content/curriculum_model")
tokenizer.save_pretrained("/content/curriculum_model")

print("\nTraining complete!")
print("Final model saved successfully.")


============ R1 ============
Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


R1 Epoch 1/3:   0%|          | 0/530 [00:00<?, ?it/s]

1       0.850427       1.159815       0.436000    0.420624       0.544966       0.301158


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Best model updated.


...

2       0.385810       1.441451       0.454167    0.408163       0.513514       0.421053


R3 Epoch 3/3:   0%|          | 0/3140 [00:00<?, ?it/s]

3       0.215373       1.789209       0.472500    0.474908       0.513053       0.418803

Loading best checkpoint from R3


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete!
Final model saved successfully.


In [ ]:
import os
import shutil

# Destination in Google Drive
DRIVE_DIR = "/content/drive/MyDrive/anli_checkpoints/albert_curriculum"
os.makedirs(DRIVE_DIR, exist_ok=True)

folders_to_copy = [
    "best_R1",
    "best_R2",
    "best_R3",
    "curriculum_model"
]

for folder in folders_to_copy:
    src = f"/content/{folder}"
    dst = f"{DRIVE_DIR}/{folder}"

    if os.path.exists(src):
        # Remove old copy if it exists
        if os.path.exists(dst):
            shutil.rmtree(dst)

        shutil.copytree(src, dst)
        print(f"✓ Copied {folder} to Google Drive")
    else:
        print(f"✗ {folder} not found")

print("\nAll available models have been copied successfully!")

✓ Copied best_R1 to Google Drive
✓ Copied best_R2 to Google Drive
✓ Copied best_R3 to Google Drive
✓ Copied curriculum_model to Google Drive

All available models have been copied successfully!


In [ ]:
from transformers import AlbertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import torch
import copy

# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Fresh model for baseline
# -------------------------
baseline_model = AlbertForSequenceClassification.from_pretrained(
    "albert-base-v2",
    num_labels=3
).to(device)

# -------------------------
# Mixed Dataset
# -------------------------
all_train = r1_data + r2_data + r3_data
all_dev   = r1_dev + r2_dev + r3_dev

print(f"Baseline training on {len(all_train)} mixed examples")

train_loader = DataLoader(
    ANLIDataset(all_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_val_loss = float("inf")
best_model_state = None

checkpoint_path = "/content/best_baseline"

print("\n========== Baseline (Mixed Training) ==========\n")

print(
    f"{'Epoch':<8}"
    f"{'Train Loss':<15}"
    f"{'Val Loss':<15}"
    f"{'Accuracy':<12}"
    f"{'F1 Entail':<15}"
    f"{'F1 Neutral':<15}"
    f"{'F1 Contra'}"
)

# ==========================
# Epoch Loop
# ==========================

for epoch in range(MAX_EPOCHS):

    baseline_model.train()

    running_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Baseline Epoch {epoch+1}/{MAX_EPOCHS}",
        dynamic_ncols=True
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    # ==========================
    # End of Epoch Evaluation
    # ==========================

    train_loss = running_loss / len(train_loader)

    val_loss, accuracy, f1 = evaluate(
        baseline_model,
        all_dev,
        device
    )

    print(
        f"{epoch+1:<8}"
        f"{train_loss:<15.6f}"
        f"{val_loss:<15.6f}"
        f"{accuracy:<12.6f}"
        f"{f1[0]:<15.6f}"
        f"{f1[1]:<15.6f}"
        f"{f1[2]:.6f}"
    )

    # ==========================
    # Save Best Model
    # ==========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            baseline_model.state_dict()
        )

        baseline_model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

        print("✓ Best model updated.")

# ==========================
# Restore Best Model
# ==========================

print("\nLoading best baseline checkpoint")

baseline_model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

baseline_model.save_pretrained("/content/drive/MyDrive/baseline_model")
tokenizer.save_pretrained("/content/drive/MyDrive/baseline_model")

print("\nBaseline training complete!")
print("Final model saved successfully.")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.bias             | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline training on 162865 mixed examples

========== Baseline (Mixed Training) ==========

Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


Baseline Epoch 1/3:   0%|          | 0/5090 [00:00<?, ?it/s]

1       0.680334       1.308622       0.440000    0.414390       0.512227       0.380139


...

✓ Best model updated.


Baseline Epoch 3/3:   0%|          | 0/5090 [00:00<?, ?it/s]

3       0.266829       1.505406       0.495937    0.500000       0.539514       0.438531

Loading best baseline checkpoint


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline training complete!
Final model saved successfully.


In [5]:
from transformers import AlbertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import torch
import copy

# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Fresh model for baseline
# -------------------------
baseline_model = AlbertForSequenceClassification.from_pretrained(
    "albert-base-v2",
    num_labels=3
).to(device)

# -------------------------
# Mixed Dataset
# -------------------------
all_train = r2_data
all_dev   = r2_dev

print(f"Baseline training on {len(all_train)} mixed examples")

train_loader = DataLoader(
    ANLIDataset(all_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_val_loss = float("inf")
best_model_state = None

checkpoint_path = "/content/best_baseline"

print("\n========== Baseline (Mixed Training) ==========\n")

print(
    f"{'Epoch':<8}"
    f"{'Train Loss':<15}"
    f"{'Val Loss':<15}"
    f"{'Accuracy':<12}"
    f"{'F1 Entail':<15}"
    f"{'F1 Neutral':<15}"
    f"{'F1 Contra'}"
)

# ==========================
# Epoch Loop
# ==========================

for epoch in range(MAX_EPOCHS):

    baseline_model.train()

    running_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Baseline Epoch {epoch+1}/{MAX_EPOCHS}",
        dynamic_ncols=True
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    # ==========================
    # End of Epoch Evaluation
    # ==========================

    train_loss = running_loss / len(train_loader)

    val_loss, accuracy, f1 = evaluate(
        baseline_model,
        all_dev,
        device
    )

    print(
        f"{epoch+1:<8}"
        f"{train_loss:<15.6f}"
        f"{val_loss:<15.6f}"
        f"{accuracy:<12.6f}"
        f"{f1[0]:<15.6f}"
        f"{f1[1]:<15.6f}"
        f"{f1[2]:.6f}"
    )

    # ==========================
    # Save Best Model
    # ==========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            baseline_model.state_dict()
        )

        baseline_model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

        print("✓ Best model updated.")

# ==========================
# Restore Best Model
# ==========================

print("\nLoading best baseline checkpoint")

baseline_model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

baseline_model.save_pretrained("/content/drive/MyDrive/baseline_model")
tokenizer.save_pretrained("/content/drive/MyDrive/baseline_model")

print("\nBaseline training complete!")
print("Final model saved successfully.")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline training on 45460 mixed examples

========== Baseline (Mixed Training) ==========

Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


Baseline Epoch 1/3:   0%|          | 0/1421 [00:00<?, ?it/s]

1       0.694692       1.387954       0.390000    0.444954       0.455587       0.172093


...

2       0.412257       1.449457       0.439000    0.450899       0.477273       0.376963


Baseline Epoch 3/3:   0%|          | 0/1421 [00:00<?, ?it/s]

3       0.234884       1.690831       0.462000    0.469444       0.501466       0.408027

Loading best baseline checkpoint


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline training complete!
Final model saved successfully.


In [6]:
from transformers import AlbertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
import torch
import copy

# -------------------------
# Training Parameters
# -------------------------
MAX_EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# -------------------------
# Fresh model for baseline
# -------------------------
baseline_model = AlbertForSequenceClassification.from_pretrained(
    "albert-base-v2",
    num_labels=3
).to(device)

# -------------------------
# Mixed Dataset
# -------------------------
all_train = r3_data
all_dev   = r3_dev

print(f"Baseline training on {len(all_train)} mixed examples")

train_loader = DataLoader(
    ANLIDataset(all_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(
    baseline_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

best_val_loss = float("inf")
best_model_state = None

checkpoint_path = "/content/best_baseline"

print("\n========== Baseline (Mixed Training) ==========\n")

print(
    f"{'Epoch':<8}"
    f"{'Train Loss':<15}"
    f"{'Val Loss':<15}"
    f"{'Accuracy':<12}"
    f"{'F1 Entail':<15}"
    f"{'F1 Neutral':<15}"
    f"{'F1 Contra'}"
)

# ==========================
# Epoch Loop
# ==========================

for epoch in range(MAX_EPOCHS):

    baseline_model.train()

    running_loss = 0

    progress_bar = tqdm(
        train_loader,
        desc=f"Baseline Epoch {epoch+1}/{MAX_EPOCHS}",
        dynamic_ncols=True
    )

    for batch in progress_bar:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = baseline_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            baseline_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    # ==========================
    # End of Epoch Evaluation
    # ==========================

    train_loss = running_loss / len(train_loader)

    val_loss, accuracy, f1 = evaluate(
        baseline_model,
        all_dev,
        device
    )

    print(
        f"{epoch+1:<8}"
        f"{train_loss:<15.6f}"
        f"{val_loss:<15.6f}"
        f"{accuracy:<12.6f}"
        f"{f1[0]:<15.6f}"
        f"{f1[1]:<15.6f}"
        f"{f1[2]:.6f}"
    )

    # ==========================
    # Save Best Model
    # ==========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            baseline_model.state_dict()
        )

        baseline_model.save_pretrained(checkpoint_path)
        tokenizer.save_pretrained(checkpoint_path)

        print("✓ Best model updated.")

# ==========================
# Restore Best Model
# ==========================

print("\nLoading best baseline checkpoint")

baseline_model.load_state_dict(best_model_state)

# ==========================
# Save Final Model
# ==========================

baseline_model.save_pretrained("/content/drive/MyDrive/baseline1_model")
tokenizer.save_pretrained("/content/drive/MyDrive/baseline1_model")

print("\nBaseline training complete!")
print("Final model saved successfully.")

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

[transformers] AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.decoder.bias     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Baseline training on 100459 mixed examples

========== Baseline (Mixed Training) ==========

Epoch   Train Loss     Val Loss       Accuracy    F1 Entail      F1 Neutral     F1 Contra


Baseline Epoch 1/3:   0%|          | 0/3140 [00:00<?, ?it/s]

1       0.709986       1.266575       0.447500    0.469322       0.478697       0.380386


...

2       0.421514       1.360801       0.442500    0.418136       0.490566       0.408511


Baseline Epoch 3/3:   0%|          | 0/3140 [00:00<?, ?it/s]

3       0.235165       1.727256       0.469167    0.469734       0.506388       0.423562

Loading best baseline checkpoint


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Baseline training complete!
Final model saved successfully.
